In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
import numpy as np
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# ========== Configuration ==========
excel_file = "smiles.xlsx"
output_file = "all_descriptors_density.csv"
bw_method = 'scott'

# ========== Descriptor calculation functions ==========
def calc_molecular_weight(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Descriptors.MolWt(mol) if mol else None
    except:
        return None

def calc_logp(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Crippen.MolLogP(mol) if mol else None
    except:
        return None

def calc_heteroatom_ratio(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            return None
        atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
        total_atoms = len(atoms)
        if total_atoms == 0:
            return 0.0
        heteroatoms = sum(1 for a in atoms if a not in [1, 6])
        return (heteroatoms / total_atoms) * 100.0
    except:
        return None

# ========== Safe density curve generation (fixes singular matrix issues) ==========
def get_density_curve_safe(data, n_points=200):
    """
    Safe density estimation function that handles cases with very small variance
    """
    if len(data) < 3:  # Too few data points
        return np.array([]), np.array([])
    
    data = np.array(data)
    
    # Check if data is nearly constant
    data_std = np.std(data)
    if data_std < 1e-6:  # Standard deviation is almost zero
        # All values are the same, return a narrow peak
        x_range = np.linspace(data[0] - 1, data[0] + 1, n_points)
        y_vals = np.zeros_like(x_range)
        # Place a spike at the data point
        idx = np.argmin(np.abs(x_range - data[0]))
        y_vals[idx] = 1.0
        # Smooth it
        from scipy.ndimage import gaussian_filter1d
        y_vals = gaussian_filter1d(y_vals, sigma=2)
        y_vals = y_vals / np.trapz(y_vals, x_range)  # Normalize
        return x_range, y_vals
    
    # Try standard KDE
    try:
        # Add tiny noise to break exact duplicates
        if len(np.unique(data)) < len(data) * 0.9:  # Many duplicate values
            noise = np.random.normal(0, data_std * 0.01, len(data))
            data = data + noise
        
        kde = gaussian_kde(data, bw_method=bw_method)
        x_min, x_max = min(data), max(data)
        # Extend range to avoid boundary truncation
        pad = max((x_max - x_min) * 0.1, 1.0)
        x_range = np.linspace(x_min - pad, x_max + pad, n_points)
        y_vals = kde(x_range)
        return x_range, y_vals
    
    except np.linalg.LinAlgError:
        # If still failing, use histogram smoothing method
        hist, bin_edges = np.histogram(data, bins=min(20, len(np.unique(data))), density=True)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        # Smooth histogram
        from scipy.ndimage import gaussian_filter1d
        y_smooth = gaussian_filter1d(hist, sigma=1)
        y_smooth = y_smooth / np.trapz(y_smooth, bin_centers)
        # Interpolate to denser points
        x_range = np.linspace(min(data), max(data), n_points)
        from scipy.interpolate import interp1d
        f = interp1d(bin_centers, y_smooth, kind='linear', fill_value=0, bounds_error=False)
        y_vals = f(x_range)
        return x_range, y_vals

def pad_to_length(arr, target_len):
    if len(arr) == 0:
        return [np.nan] * target_len
    return np.pad(arr, (0, target_len - len(arr)), constant_values=np.nan)

# ========== Batch calculation ==========
def calculate_descriptors(smiles_list, func, desc_name):
    results = []
    for i, smi in enumerate(smiles_list):
        if i % 500 == 0 and i > 0:
            print(f"    Calculated {i}/{len(smiles_list)}")
        val = func(smi)
        if val is not None:
            results.append(val)
    return results

# ========== Main program ==========
print("="*70)
print("Molecular Descriptor Calculation (Fixed Version)")
print("="*70)

# Read data
print(f"\n1. Reading file: {excel_file}")
df = pd.read_excel(excel_file)
train_smiles = df['TRAIN'].dropna().tolist()
test_smiles = df['TEST'].dropna().tolist()

print(f"   TRAIN samples: {len(train_smiles)}")
print(f"   TEST samples: {len(test_smiles)}")

# Define descriptors (only 3)
descriptors = {
    'molecular_weight': ('Molecular Weight (Da)', calc_molecular_weight),
    'logp': ('LogP', calc_logp),
    'heteroatom_ratio': ('Heteroatom Ratio (%)', calc_heteroatom_ratio)
}

# Calculate descriptors
print("\n2. Calculating descriptors...")
all_results = {}

for desc_key, (desc_name, func) in descriptors.items():
    print(f"\n   {desc_name}:")
    print(f"     TRAIN set...")
    train_vals = calculate_descriptors(train_smiles, func, desc_name)
    print(f"     TEST set...")
    test_vals = calculate_descriptors(test_smiles, func, desc_name)
    
    all_results[desc_key] = {
        'name': desc_name,
        'train': train_vals,
        'test': test_vals
    }
    
    print(f"     Valid values: TRAIN={len(train_vals)}, TEST={len(test_vals)}")
    if len(train_vals) > 0:
        print(f"     Range: TRAIN=[{min(train_vals):.2f}, {max(train_vals):.2f}], "
              f"Std={np.std(train_vals):.4f}")

# ========== Generate density curves ==========
print("\n3. Generating density curves...")
output_dfs = []

# Process main descriptors
for desc_key, data in all_results.items():
    print(f"   Processing {data['name']}...")
    
    # Generate density curves
    x_train, y_train = get_density_curve_safe(data['train'])
    x_test, y_test = get_density_curve_safe(data['test'])
    
    max_len = max(len(x_train), len(x_test))
    
    if max_len == 0:
        print(f"     Warning: {data['name']} insufficient data, skipping")
        continue
    
    df_desc = pd.DataFrame({
        f'{desc_key}_Train_X': pad_to_length(x_train, max_len),
        f'{desc_key}_Train_Y': pad_to_length(y_train, max_len),
        f'{desc_key}_Test_X': pad_to_length(x_test, max_len),
        f'{desc_key}_Test_Y': pad_to_length(y_test, max_len)
    })
    output_dfs.append(df_desc)
    print(f"     Done, {max_len} data points")

# Merge output
if output_dfs:
    print("\n4. Saving results...")
    final_df = pd.concat(output_dfs, axis=1)
    final_df.to_csv(output_file, index=False)
    print(f"    Generated: {output_file}")
else:
    print("\nError: No density curves were successfully generated!")
    exit(1)

# ========== Statistics summary ==========
print("\n" + "="*70)
print("Statistics Summary")
print("="*70)

for desc_key, data in all_results.items():
    desc_name = data['name']
    train_vals = data['train']
    test_vals = data['test']
    
    print(f"\n{desc_name}:")
    if len(train_vals) > 0:
        print(f"  TRAIN: Mean={np.mean(train_vals):.2f}, "
              f"Median={np.median(train_vals):.2f}, "
              f"Std={np.std(train_vals):.2f}, "
              f"Range=[{min(train_vals):.2f}, {max(train_vals):.2f}]")
    if len(test_vals) > 0:
        print(f"  TEST:  Mean={np.mean(test_vals):.2f}, "
              f"Median={np.median(test_vals):.2f}, "
              f"Std={np.std(test_vals):.2f}, "
              f"Range=[{min(test_vals):.2f}, {max(test_vals):.2f}]")

# ========== Generate preview plot ==========
try:
    import matplotlib.pyplot as plt
    
    n_plots = len(descriptors)  # Only 3 plots
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    plot_idx = 0
    
    # Plot main descriptors
    for desc_key, data in all_results.items():
        ax = axes[plot_idx]
        desc_name = data['name']
        
        x_train, y_train = get_density_curve_safe(data['train'])
        x_test, y_test = get_density_curve_safe(data['test'])
        
        if len(x_train) > 0:
            ax.plot(x_train, y_train, label='TRAIN', linewidth=2)
        if len(x_test) > 0:
            ax.plot(x_test, y_test, label='TEST', linewidth=2)
        
        ax.set_xlabel(desc_name)
        ax.set_ylabel('Density')
        ax.set_title(desc_name)
        ax.legend()
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    # Hide unused subplots (if any)
    for idx in range(plot_idx, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('all_descriptors_preview.png', dpi=150, bbox_inches='tight')
    print("\n✓ Generated preview plot: all_descriptors_preview.png")
    
except Exception as e:
    print(f"\nPreview plot generation failed: {e}")

print("\n" + "="*70)
print("Done! Import the CSV file into Origin to plot density curves")
print(f"Output file: {output_file}")
print("="*70)